# Run the assignment
Runtime → Change runtime type → **T4 GPU** first. Then use Runtime → **Run all**.

It asks for two things near the start: Google Drive access, and your W&B API key (from https://wandb.ai/authorize). After that it needs about 1-2 hours; keep the tab open.

Finished files are copied to `MyDrive/CS25M540_SEDL_Assignment1_VGG6` in Google Drive. If the session disconnects, W&B keeps the finished runs. Do not press Run all again, because that repeats them.

In [ ]:
# Stop early if there is no GPU: on CPU this takes days, not hours
import os
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.'
print(torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CS25M540_SEDL_Assignment1_VGG6'
os.makedirs(OUT, exist_ok=True)

In [ ]:
%cd /content
!test -d CS25M540_SEDL_Assignment1_VGG6 || git clone https://github.com/tanujgupta12/CS25M540_SEDL_Assignment1_VGG6.git
%cd /content/CS25M540_SEDL_Assignment1_VGG6
!pip install -q -r requirements.txt

In [ ]:
# Paste your W&B API key when asked (https://wandb.ai/authorize)
!wandb login

In [ ]:
# Q1 baseline (Figure 1 uses checkpoints/baseline/curves.png)
!python train.py --activation relu --optimizer nesterov --epochs 10 --save_dir checkpoints/baseline
!cp -r checkpoints/baseline "$OUT"/

In [ ]:
# Q2/Q3: all three sweeps, 17 runs
!bash run_sweeps.sh CS6886W-SEDL-Assignment1-VGG6

In [ ]:
# Q4: retrain the best sweep configuration (highest validation accuracy) for 30 epochs
best = !python wandb_results.py best
print('Best configuration:', best[-1])
!python train.py {best[-1]} --epochs 30 --seed 42 --save_dir checkpoints/best
!python evaluate.py --checkpoint checkpoints/best/best.pt | tee checkpoints/best/evaluate.txt

In [ ]:
# Export everything the report needs, and record the environment
!python wandb_results.py export --out results.json
import sys
import torchvision
import wandb
with open('environment.txt', 'w') as f:
    f.write(f'Python {sys.version.split()[0]}\nPyTorch {torch.__version__}\n'
            f'torchvision {torchvision.__version__}\nwandb {wandb.__version__}\n'
            f'GPU {torch.cuda.get_device_name(0)}\n')
!cat environment.txt
!cp -r checkpoints/best "$OUT"/
!cp results.json environment.txt "$OUT"/
!zip -q -r cs25m540_results.zip results.json environment.txt checkpoints/baseline checkpoints/best
!cp cs25m540_results.zip "$OUT"/

In [ ]:
# Download the results bundle (also saved in Google Drive)
from google.colab import files
files.download('cs25m540_results.zip')